# Notebook 01: Data Acquisition and Documentation

**Project:** Detecting Healthcare Fraud, Waste, and Abuse in Medicare Claims

**Author:** Aliance Tedonfouet

The study period is billing years 2019 to 2023. This notebook confirms my data files are in place, checks that the CMS columns I need exist, collapses the CMS file to one row per provider per year, and measures how many providers with a fraud-related exclusion can be matched to Medicare billing data and labeled as positive cases. No models are built here.

## 1. Configuration

**Study period: 2019 to 2023.** I started with 2023 as a first test year. I later added 2019 to 2022 because a provider who billed in an earlier year and was excluded later is a valid positive case for that earlier year, which increases the number of positives. I am not using 2024 because exclusions for 2024 billing may not be recorded yet, so some fraud there would be labeled negative. 2020 and 2021 billing was affected by COVID-19, so I will record the year as a variable and check results with and without those years.

**LEIE:** the October 2026 current-exclusions file.

**Label rule:** a provider-year is positive if the provider's earliest fraud-coded exclusion falls after the billing year and within LOOKAHEAD_YEARS years of it. I set the window to 3 years as a starting point. Section 7 shows how the number of positives changes with a window of 1 to 5 years.

FRAUD_EXCLUSION_TYPES is my working definition of "fraud-related": 1128(a)(1), 1128(a)(3), 1128(b)(1), and 1128(b)(7). I compared each code to the OIG's published exclusion authorities on 11/21/2026. Two limits: 1128(b)(1) combines misdemeanor health care fraud with fraud in non-health-care programs, and 1128(b)(7) covers "other prohibited activities" beyond billing fraud. This choice changes who counts as a positive case, so Phase 4 will test how results change if the definition is widened or narrowed.

In [1]:
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_EXT = PROJECT_ROOT / "data" / "external"
DATA_PROC = PROJECT_ROOT / "data" / "processed"
TABLES = PROJECT_ROOT / "results" / "tables"
for folder in (DATA_PROC, TABLES):
    folder.mkdir(parents=True, exist_ok=True)

CMS_FILE = DATA_RAW / "cms_partb_2019_2023.csv"
LEIE_FILE = DATA_EXT / "hhs_leie_current.csv"

STUDY_YEARS = [2019, 2020, 2021, 2022, 2023]
LOOKAHEAD_YEARS = 3   # provisional: exclusion within 3 years AFTER the billing year

CMS_COLUMNS_WANTED = [
    "year", "Rndrng_NPI", "Rndrng_Prvdr_Ent_Cd", "Rndrng_Prvdr_Type",
    "Rndrng_Prvdr_State_Abrvtn", "HCPCS_Cd", "Tot_Srvcs", "Tot_Benes",
    "Avg_Sbmtd_Chrg", "Avg_Mdcr_Alowd_Amt", "Avg_Mdcr_Pymt_Amt",
]

# Working definition of "fraud-related" (see the markdown above and the README)
FRAUD_EXCLUSION_TYPES = {"1128a1", "1128a3", "1128b1", "1128b7"}

print("Project root:", PROJECT_ROOT)
print("pandas", pd.__version__, "| run at", datetime.now().strftime("%Y-%m-%d %H:%M"))

Project root: /Users/ted/Documents/Medicare-Fraud-Detection-Research
pandas 3.1.1 | run at 2026-11-21 02:46


## 2. Checking the CMS column names

CMS has changed column names between releases, so I read only the header row and compared it to the columns I plan to use. The combined 2019 to 2023 file has a year column, which is what lets me assign each row to a billing year.

In [2]:
missing = []
if CMS_FILE.exists():
    header = pd.read_csv(CMS_FILE, nrows=0).columns.tolist()
    print(f"File has {len(header)} columns:\n")
    print(header)
    missing = [c for c in CMS_COLUMNS_WANTED if c not in header]
    print("\nWanted columns missing from file:", missing if missing else "none")
    if missing:
        print("Update CMS_COLUMNS_WANTED above to match the real names, then re-run.")
else:
    print("CMS file not found. Download it first (see docs/DOWNLOAD_LOG.md).")

File has 29 columns:

['year', 'Rndrng_NPI', 'Rndrng_Prvdr_Last_Org_Name', 'Rndrng_Prvdr_First_Name', 'Rndrng_Prvdr_MI', 'Rndrng_Prvdr_Crdntls', 'Rndrng_Prvdr_Ent_Cd', 'Rndrng_Prvdr_St1', 'Rndrng_Prvdr_St2', 'Rndrng_Prvdr_City', 'Rndrng_Prvdr_State_Abrvtn', 'Rndrng_Prvdr_State_FIPS', 'Rndrng_Prvdr_Zip5', 'Rndrng_Prvdr_RUCA', 'Rndrng_Prvdr_RUCA_Desc', 'Rndrng_Prvdr_Cntry', 'Rndrng_Prvdr_Type', 'Rndrng_Prvdr_Mdcr_Prtcptg_Ind', 'HCPCS_Cd', 'HCPCS_Desc', 'HCPCS_Drug_Ind', 'Place_Of_Srvc', 'Tot_Benes', 'Tot_Srvcs', 'Tot_Bene_Day_Srvcs', 'Avg_Sbmtd_Chrg', 'Avg_Mdcr_Alowd_Amt', 'Avg_Mdcr_Pymt_Amt', 'Avg_Mdcr_Stdzd_Amt']

Wanted columns missing from file: none


## 3. CMS inventory and the provider-year table

The file has about 49 million rows (one per provider, procedure code, and place of service, per year), which is too large to load at once. I read it in chunks of one million rows. In the same pass I count rows and missing values, and I collapse the rows to one row per provider per year. That table is what the labeling in Section 6 uses.

Each provider-year row holds the number of procedure-code rows, total services, and estimated allowed, paid, and submitted amounts (average amount times services). These are placeholders for Phase 2 feature engineering, not final features.

Rows and distinct provider NPIs per year are shown in the output below and saved to `results/tables/inventory_cms_by_year.csv`. None of the columns I use have missing values.

In [3]:
parts = []
rows_per_year = {}
nulls = pd.Series(0, index=CMS_COLUMNS_WANTED, dtype="int64")
total_rows = 0

if CMS_FILE.exists() and not missing:
    reader = pd.read_csv(
        CMS_FILE, usecols=CMS_COLUMNS_WANTED, chunksize=1_000_000,
        dtype={"Rndrng_NPI": "string", "HCPCS_Cd": "string"},
    )
    for i, chunk in enumerate(reader):
        total_rows += len(chunk)
        nulls += chunk.isna().sum()
        for y, n in chunk["year"].value_counts().items():
            rows_per_year[int(y)] = rows_per_year.get(int(y), 0) + int(n)

        chunk["est_allowed"] = chunk["Tot_Srvcs"] * chunk["Avg_Mdcr_Alowd_Amt"]
        chunk["est_paid"] = chunk["Tot_Srvcs"] * chunk["Avg_Mdcr_Pymt_Amt"]
        chunk["est_submitted"] = chunk["Tot_Srvcs"] * chunk["Avg_Sbmtd_Chrg"]

        g = (chunk.groupby(["year", "Rndrng_NPI"])
                  .agg(n_rows=("HCPCS_Cd", "size"),
                       tot_srvcs=("Tot_Srvcs", "sum"),
                       est_allowed=("est_allowed", "sum"),
                       est_paid=("est_paid", "sum"),
                       est_submitted=("est_submitted", "sum"),
                       ent_cd=("Rndrng_Prvdr_Ent_Cd", "first"),
                       prvdr_type=("Rndrng_Prvdr_Type", "first"),
                       state=("Rndrng_Prvdr_State_Abrvtn", "first"))
                  .reset_index())
        parts.append(g)
        if i % 10 == 0:
            print(f"chunk {i} done")

    # A provider can span two chunks, so combine the partial results
    prov = (pd.concat(parts)
              .groupby(["year", "Rndrng_NPI"])
              .agg(n_rows=("n_rows", "sum"), tot_srvcs=("tot_srvcs", "sum"),
                   est_allowed=("est_allowed", "sum"), est_paid=("est_paid", "sum"),
                   est_submitted=("est_submitted", "sum"),
                   ent_cd=("ent_cd", "first"), prvdr_type=("prvdr_type", "first"),
                   state=("state", "first"))
              .reset_index())
    del parts

    assert set(rows_per_year) == set(STUDY_YEARS), f"Unexpected years in file: {sorted(rows_per_year)}"

    cms_inventory = pd.DataFrame({"missing_n": nulls,
                                  "missing_pct": (nulls / total_rows * 100).round(2)})
    year_inventory = pd.DataFrame({
        "rows": pd.Series(rows_per_year).sort_index(),
        "distinct_npis": prov.groupby("year")["Rndrng_NPI"].nunique(),
    })
    cms_inventory.to_csv(TABLES / "inventory_cms_columns.csv")
    year_inventory.to_csv(TABLES / "inventory_cms_by_year.csv")
    prov.to_csv(DATA_PROC / "provider_year.csv", index=False)   # data/ is not committed to GitHub

    print(f"\nCMS rows in total: {total_rows:,}")
    print(f"Provider-year rows: {len(prov):,}\n")
    print(year_inventory, "\n")
    print(cms_inventory)
else:
    print("CMS file not found, or columns are missing. Fix Section 2 first.")

chunk 0 done
chunk 10 done
chunk 20 done
chunk 30 done
chunk 40 done

CMS rows in total: 48,891,840
Provider-year rows: 5,626,423

          rows  distinct_npis
2019  10140228        1093367
2020   9449361        1085313
2021   9886177        1123589
2022   9755427        1148873
2023   9660647        1175281 

                           missing_n  missing_pct
year                               0          0.0
Rndrng_NPI                         0          0.0
Rndrng_Prvdr_Ent_Cd                0          0.0
Rndrng_Prvdr_Type                  0          0.0
Rndrng_Prvdr_State_Abrvtn          0          0.0
HCPCS_Cd                           0          0.0
Tot_Srvcs                          0          0.0
Tot_Benes                          0          0.0
Avg_Sbmtd_Chrg                     0          0.0
Avg_Mdcr_Alowd_Amt                 0          0.0
Avg_Mdcr_Pymt_Amt                  0          0.0


## 4. LEIE records and NPI cleaning

In the LEIE, providers without a usable 10-digit NPI are commonly stored as zeros or left blank.
Those rows cannot be matched to Medicare billing data, so they are set aside here.
The file I downloaded lists current exclusions only, so providers who were later reinstated are not in it.
A provider that was excluded and then reinstated would appear as a negative case. I am recording this as a limitation.

In [4]:
leie = None
if LEIE_FILE.exists():
    leie = pd.read_csv(LEIE_FILE, dtype=str, low_memory=False)
    print("LEIE columns:", leie.columns.tolist())
    print(f"LEIE records: {len(leie):,}")

    npi = leie["NPI"].fillna("").str.strip()
    leie["npi_clean"] = npi.where(npi.str.fullmatch(r"\d{10}") & (npi != "0000000000"))
    leie["code_clean"] = (leie["EXCLTYPE"].fillna("").str.lower()
                          .str.replace(r"[^a-z0-9]", "", regex=True))
    print(f"Records with a usable NPI: {leie['npi_clean'].notna().sum():,}")

LEIE columns: ['LASTNAME', 'FIRSTNAME', 'MIDNAME', 'BUSNAME', 'GENERAL', 'SPECIALTY', 'UPIN', 'NPI', 'DOB', 'ADDRESS', 'CITY', 'STATE', 'ZIP', 'EXCLTYPE', 'EXCLDATE', 'REINDATE', 'WAIVERDATE', 'WVRSTATE']
LEIE records: 84,224
Records with a usable NPI: 9,023


## 5. Exclusion-type distribution

This section helps me see which exclusion codes appear in the LEIE and how common each one is. I used the OIG's page on exclusion authorities for the codes in my definition. A few codes in the file (1128aa, brchsa, brchcia, 1160) are not described on the OIG authorities page.

In [5]:
if leie is not None:
    dist = (leie.groupby("code_clean")
                .agg(records=("code_clean", "size"), with_npi=("npi_clean", lambda s: s.notna().sum()))
                .sort_values("records", ascending=False))
    dist["in_fraud_definition"] = dist.index.isin(FRAUD_EXCLUSION_TYPES)
    dist.to_csv(TABLES / "leie_exclusion_type_distribution.csv")
    print(dist.head(25))

            records  with_npi  in_fraud_definition
code_clean                                        
1128b4        33463      2691                False
1128a1        26104      3413                 True
1128a2         8127       404                False
1128a3         5884       684                 True
1128a4         3592      1019                False
1128b14        2164       196                False
1128b8         1497        96                False
1128b1          943        48                 True
1128b5          879       156                False
1128b7          766       208                 True
1128b3          329        20                False
1128aa          178        37                False
1128b6           77         9                False
1128b2           65         8                False
1156             53         1                False
1128b15          37         1                False
brchsa           28        20                False
brchcia          14         9  

## 6. Building the labels

A provider who is excluded stops billing Medicare, so matching the current LEIE to a single billing year finds very few cases. The study therefore asks a forward-looking question: **among providers who billed in year Y, who was excluded for a fraud-related reason after Y?**

Rules, applied to each provider-year:

- Each NPI is assigned its **earliest** fraud-coded exclusion year.
- **Positive (label = 1):** that exclusion year is after the billing year and no more than `LOOKAHEAD_YEARS` later.
- **Already excluded:** the exclusion year is the billing year or earlier. The timing is ambiguous (billing after exclusion, or exclusion partway through the year), so these rows are flagged and will be kept out of modeling.
- **Negative (label = 0):** everyone else, including providers whose exclusion falls outside the window. Rows flagged as already excluded also have label 0 in the table but are kept out of modeling.

First I look at when the fraud-coded exclusions happened.

In [6]:
fraud = leie[leie["code_clean"].isin(FRAUD_EXCLUSION_TYPES) & leie["npi_clean"].notna()].copy()
fraud["excl_year"] = pd.to_numeric(fraud["EXCLDATE"].str[:4], errors="coerce")

print(f"Fraud-coded LEIE records with a usable NPI: {len(fraud):,}")
print(f"Unique NPIs: {fraud['npi_clean'].nunique():,}")
print(f"Records with a missing or unreadable exclusion date: {fraud['excl_year'].isna().sum()}\n")
print("Exclusion year of fraud-coded records:")
print(fraud["excl_year"].dropna().astype(int).value_counts().sort_index().to_string())

Fraud-coded LEIE records with a usable NPI: 4,353
Unique NPIs: 4,311
Records with a missing or unreadable exclusion date: 0

Exclusion year of fraud-coded records:
excl_year
1988      2
1993      1
1995      2
1996      1
1998      1
2000      1
2001      1
2002      2
2005      1
2006      3
2007      7
2008     12
2009     91
2010     95
2011    147
2012    159
2013    147
2014    206
2015    221
2016    254
2017    266
2018    218
2019    251
2020    258
2021    218
2022    321
2023    349
2024    438
2025    354
2026    326


In [7]:
# One row per NPI: the earliest fraud-coded exclusion year
first_excl = (fraud.groupby("npi_clean")["excl_year"].min()
                   .rename("excl_year").reset_index()
                   .rename(columns={"npi_clean": "Rndrng_NPI"}))
first_excl["Rndrng_NPI"] = first_excl["Rndrng_NPI"].astype("string")

df = prov.merge(first_excl, on="Rndrng_NPI", how="left")

def make_label(frame, window):
    """1 if the earliest fraud-coded exclusion is after the billing year and within `window` years."""
    return ((frame["excl_year"] > frame["year"]) &
            (frame["excl_year"] <= frame["year"] + window)).astype(int)

df["already_excluded"] = df["excl_year"] <= df["year"]
df["label"] = make_label(df, LOOKAHEAD_YEARS)

print(f"Provider-year rows: {len(df):,}")
print(f"Positive provider-years: {int(df['label'].sum()):,}")
print(f"Unique positive providers: {df.loc[df['label'] == 1, 'Rndrng_NPI'].nunique():,}")

Provider-year rows: 5,626,423
Positive provider-years: 588
Unique positive providers: 340


## 7. Phase 1 results: how many labeled providers do I have?

This determines whether a supervised model is feasible at all and how severe the class imbalance will be.

With a 3-year window there are **340 unique positive providers** (588 positive provider-years) out of about 5.6 million provider-years, roughly 1 in 8,000 to 1 in 12,000 per year. Counting provider-years overstates the sample, because the same provider can be positive in several years. The unique count is the sample size, and cross-validation will be split by NPI.

The positives are almost all individuals (332 of 340). The specialty table below shows counts, not rates. Primary care is also the largest group of all providers, so it says nothing yet about which specialties are riskier.

In [8]:
label_summary = (df.groupby("year")
                   .agg(providers=("Rndrng_NPI", "nunique"),
                        positives=("label", "sum"),
                        already_excluded=("already_excluded", "sum")))
label_summary["rate_per_10k"] = (label_summary["positives"] / label_summary["providers"] * 10_000).round(2)
print(label_summary, "\n")
label_summary.to_csv(TABLES / "label_summary_by_year.csv")

pos = df[df["label"] == 1]
pos_unique = pos.drop_duplicates("Rndrng_NPI")
print("Entity type of unique positives (I = individual, O = organization):")
print(pos_unique["ent_cd"].value_counts(), "\n")
print("Top specialties among unique positives (counts, not rates):")
print(pos_unique["prvdr_type"].value_counts().head(10), "\n")

# Sensitivity of the sample size to the look-ahead window
rows = []
for window in [1, 2, 3, 4, 5]:
    lab = make_label(df, window).astype(bool)
    rows.append({"window_years": window,
                 "positive_provider_years": int(lab.sum()),
                 "unique_positive_NPIs": int(df.loc[lab, "Rndrng_NPI"].nunique())})
window_sensitivity = pd.DataFrame(rows)
print(window_sensitivity)
window_sensitivity.to_csv(TABLES / "label_window_sensitivity.csv", index=False)

      providers  positives  already_excluded  rate_per_10k
year                                                      
2019    1093367        118                 6          1.08
2020    1085313        119                 5          1.10
2021    1123589        142                 6          1.26
2022    1148873        113                 7          0.98
2023    1175281         96                 9          0.82 

Entity type of unique positives (I = individual, O = organization):
ent_cd
I    332
O      8
Name: count, dtype: int64 

Top specialties among unique positives (counts, not rates):
prvdr_type
Internal Medicine                         52
Family Practice                           51
Nurse Practitioner                        29
Chiropractic                              17
Podiatry                                  15
Physical Therapist in Private Practice    14
Emergency Medicine                        13
General Practice                          12
Physical Medicine and Rehabilitat

In [9]:
# Save the Phase 1 summary used in the README
phase1_summary = {
    "run_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "study_years": STUDY_YEARS,
    "lookahead_years": LOOKAHEAD_YEARS,
    "fraud_exclusion_types": sorted(FRAUD_EXCLUSION_TYPES),
    "cms_rows_total": int(total_rows),
    "provider_year_rows": int(len(df)),
    "leie_records": int(len(leie)),
    "leie_fraud_coded_records": int(leie["code_clean"].isin(FRAUD_EXCLUSION_TYPES).sum()),
    "leie_fraud_coded_with_usable_npi": int(len(fraud)),
    "positive_provider_years": int(df["label"].sum()),
    "unique_positive_npis": int(pos["Rndrng_NPI"].nunique()),
    "already_excluded_provider_years": int(df["already_excluded"].sum()),
    "positives_by_year": {int(k): int(v) for k, v in label_summary["positives"].items()},
    "unique_positives_by_window": {int(r.window_years): int(r.unique_positive_NPIs)
                                   for r in window_sensitivity.itertuples()},
}
with open(TABLES / "phase1_summary.json", "w") as f:
    json.dump(phase1_summary, f, indent=2)
for k, v in phase1_summary.items():
    print(f"{k:34s} {v}")
print("\nSaved results/tables/phase1_summary.json")

run_at                             2026-11-21 02:47:32
study_years                        [2019, 2020, 2021, 2022, 2023]
lookahead_years                    3
fraud_exclusion_types              ['1128a1', '1128a3', '1128b1', '1128b7']
cms_rows_total                     48891840
provider_year_rows                 5626423
leie_records                       84224
leie_fraud_coded_records           33697
leie_fraud_coded_with_usable_npi   4353
positive_provider_years            588
unique_positive_npis               340
already_excluded_provider_years    33
positives_by_year                  {2019: 118, 2020: 119, 2021: 142, 2022: 113, 2023: 96}
unique_positives_by_window         {1: 102, 2: 222, 3: 340, 4: 410, 5: 449}

Saved results/tables/phase1_summary.json


## 8. Known data issues (recorded before any modeling)

| Issue | Why it matters | Planned handling |
|---|---|---|
| Few positives (about 1 in 10,000 provider-years) | Limits supervised learning and makes metrics unstable | Use billing years 2019 to 2023 (340 unique positive providers); report variance across folds |
| COVID-19 distortion in 2020 and 2021 | Billing patterns shifted for reasons unrelated to fraud | Record year as a variable; compare with and without those years |
| Many LEIE rows lack an NPI | Cannot be matched, so the labeled set shrinks | Counted and reported |
| Same provider in several years | Counts the same fraud case more than once; can leak between training and test sets | Report unique positive providers; split cross-validation by NPI |
| LEIE reflects detected fraud only | "Negative" providers include undetected fraud | Stated in README; discussed in Phase 5 |
| Exclusion date vs. billing year | A provider may bill normally before exclusion | Positive only if excluded after the billing year; "already excluded" rows kept out of modeling |
| Right-censoring of recent years | Exclusions for 2022 and 2023 billing may not be recorded yet, so some positives are missed | State it; compare against shorter windows in Phase 4 |
| Reinstated providers are missing from the file | Some past exclusions look like negatives | Stated as a limitation |
| Look-ahead window length | Changes who is labeled positive | Fixed 3-year window; sensitivity shown in Section 7 and tested in Phase 4 |
| Fraud-code definition is a judgment call | Changes who is labeled positive; 1128(b)(1) and (b)(7) also cover some non-billing conduct | Sensitivity test in Phase 4 |
| Some LEIE codes are undocumented (1128aa, brchsa, brchcia, 1160) | Cannot tell whether they are fraud-related | Left out of the definition; counts reported |
| Medicare fee-for-service only | May not generalize to Medicare Advantage, commercial, or Medicaid claims | Limitation stated in Phase 5 |